# Train your model

Install the checkout's notebook dependencies first, following the README's
**Run notebooks locally** instructions. Select the `GreatAI (local)` kernel.
The default run uses a balanced subset of the bundled MAG data for speed.
Set `GREAT_AI_NOTEBOOK_FULL=1` before starting Jupyter to use the full splits.
Local subset results differ from the historical full-dataset results below.

## Local setup

Use Python 3.12 or 3.13 and install this checkout with `uv pip install --editable '.[notebooks]'`.
See **Run notebooks locally** in the repository README for the kernel, execution order,
and full-data options. Generated artifacts stay under `.cache/notebooks/`.


In [ ]:
import os
import sys
from pathlib import Path

# Works when opened from the repository root or any notebook directory.
_start = Path(globals().get("__file__", Path.cwd())).resolve()
ROOT = next(
    p
    for p in [_start, *_start.parents]
    if (p / "pyproject.toml").is_file() and (p / "great_ai").is_dir()
)
sys.path.insert(0, str(ROOT / "docs"))
from notebook_support import setup_notebook, mag_lines

FULL_RUN = os.environ.get("GREAT_AI_NOTEBOOK_FULL") == "1"
WORK_DIR = setup_notebook(ROOT, "tutorial")
DATA_DIR = ROOT / "docs/tutorial/data"

In [ ]:
# Dependencies are installed once in the environment; no in-kernel installation needed.

First, we have to get some data. After downloading it [from here](https://github.com/allenai/scibert/tree/master/data/text_classification/mag), we might notice that the dataset is in [JSON Lines](https://jsonlines.org/) format (each line is a separate JSON document). 

Let's write a function which takes a single line and returns the sentence and the corresponding label from it. Before returning, the sentence is also [cleaned](https://great-ai.scoutinscience.com/reference/utilities/#great_ai.utilities.clean.clean) to remove any LaTeX, XML, unicode, PDF-extraction artifacts.

In [ ]:
import json
from great_ai.utilities import clean


def preprocess(line):
    data_point = json.loads(line)

    sentence = data_point["text"]
    label = data_point["label"]

    return clean(sentence), label

Now, we can load the dataset and extract the *training* samples from it. Since we're impatient, we can do it in parallel using the [`simple_parallel_map`](https://great-ai.scoutinscience.com/reference/utilities/#great_ai.utilities.simple_parallel_map) function.

> Open files in Python are iterable: in text mode, each iteration returns the next line.

In [ ]:
from great_ai.utilities import simple_parallel_map

training_lines = mag_lines(DATA_DIR / "train.txt", None if FULL_RUN else 100)
training_data = simple_parallel_map(preprocess, training_lines, concurrency=2)
X_train = [d[0] for d in training_data]
y_train = [d[1] for d in training_data]

Let's do the same for the *test* data.

In [ ]:
test_lines = mag_lines(DATA_DIR / "test.txt", None if FULL_RUN else 30)
test_data = simple_parallel_map(preprocess, test_lines, concurrency=2)
X_test = [d[0] for d in test_data]
y_test = [d[1] for d in test_data]

After obtaining some clean data, it's time to create and train a model on it. We are going to use [scikit-learn](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html) for this purpose.

In this case, we opted for a [linear Support Vector Machine](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html) because it's known to be an adequate baseline for simple classification tasks.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

model = make_pipeline(TfidfVectorizer(), LinearSVC(random_state=42))
# todo: hyperparameter-optimisation
model.fit(X_train, y_train)

It's already finished. Let's see how well it performs. But first, we need to [configure matplotlib](https://matplotlib.org/stable/tutorials/introductory/customizing.html) to make the charts more legible. 
> `ConfusionMatrixDisplay` relies on `matplotlib` for rendering.

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (30, 15)
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["font.size"] = 12

Next, we check the quality of the model on the `test` split. We can use the classification report to check the common metrics, such as the macro F1-score. We also draw the confusion matrix to get some insights into the types of mistakes being made by the model.

In [ ]:
from sklearn import metrics

y_predicted = model.predict(X_test)

print(metrics.classification_report(y_test, y_predicted))
metrics.ConfusionMatrixDisplay.from_predictions(y_test, y_predicted)
None

The recorded run achieves a test macro F1-score of about **0.69** on the MAG sentence-classification dataset, above the historical SciBERT baseline of **0.6571** associated with the [2019 SciBERT work](https://arxiv.org/abs/1903.10676). This compares a linear SVM with one historical baseline; it does not establish current state-of-the-art performance. A broader claim would require a current benchmark with comparable data splits, preprocessing, and metrics.

You might wonder that *"this is great, but besides some utility functions (`clean`, `simple_parallel_map`, ...) what more value does GreatAI add?"*. This would be a valid argument because the scope of GreatAI actually only starts here.

> Not coincidentally, this is the point where the scope of Data Science ends but it's still a grey zone for software engineering.

In order to use this model in production, we have to make it available on some possibly shared infrastructure.

In [ ]:
from great_ai import save_model

save_model(model, key="my-domain-predictor")

### [Let's continue by finishing the deployment in the next notebook.](https://great-ai.scoutinscience.com/tutorial/deploy)